# ESEEM: an electron spin-1/2 coupled to a nuclear spin-1

This notebook simulates Hahn echo and three-pulse electron spin echo envelope modulation
(ESEEM) experiments on a electron spin-1/2 which is hyperfine coupled to a nuclear spin-1, e.g. $^{14}\mathrm{N}$.

The nuclear quadrupole interaction mixes nuclear states and produces
echo modulation. The notebook covers:
- Spin Hamiltonian with quadrupole moment
- Hahn echo
- three-pulse ESEEM

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from qutip import Qobj

from spinfinity import CC, SpinSys, hamil
from spinfinity import helperfunctions as hf

## Static spin Hamiltonian

Spin 0 is the electron; spin 1 is the nitrogen-14 nucleus.
The magnetic field is aligned with the quadrupole principal $z$ axis.

In frequency units,

$$
\frac{H}{h}
=
-\nu_e S_z-\nu_N I_z
+A_{\mathrm{iso}}\mathbf S\cdot\mathbf I
+K\left[3I_z^2-I(I+1)+\eta(I_x^2-I_y^2)\right]
$$

We use $B_0=0.5$ T, $A_{\mathrm{iso}}=-6.5$ MHz,
$K=1$ MHz, and $\eta=1$.

For $I=1$, the conventional quadrupole coupling constant is
$\chi=e^2qQ/h=4K=4$ MHz. It combines the nuclear quadrupole moment
with the local electric field gradient.

Spinfinity instead stores the coefficients of $I_x^2$, $I_y^2$,
and $I_z^2$ in `Qvector`, in meV:

$$
(Q_{xx},Q_{yy},Q_{zz})/h
=
K(\eta-1,-\eta-1,2).
$$

Thus this example reproduces the older tensor
$(Q_{xx},Q_{yy},Q_{zz})/h=(0,-2,2)$ MHz.
The old parameter `Qzz = 2 MHz` is not the conventional coupling
constant $\chi$.

In [ ]:
spin = SpinSys([0.5, 1.0])
spin.ReadoutSpin = 0

# Magnetic Field
B0 = 0.5  # T
spin.B = [0.0, 0.0, B0]
spin.BTip = [0.0, 0.0, 0.0]

# g-factors
ge = 2.0023
gamma_n = 3.077e-3  # ordinary 14N gyromagnetic ratio, GHz/T
spin.gfactorvector = [
    ge,
    gamma_n / (spin.muB * spin.meVtoGHzConversion),
]

# Hyperfine cpoupling
A_iso = -6.5e-3  # GHz
spin.Avector[0] = np.full(3, A_iso) * spin.GHztomeVConversion

# Quadrupole parameter
K = 1.0e-3  # GHz; chi = e^2 q Q / h = 4*K for I=1
eta = 1.0
spin.Qvector[1] = (
    K * np.array([eta - 1, -eta - 1, 2.0]) * spin.GHztomeVConversion
)

hamil.calc_EigEnergies(spin, rebuildH=True)
hamil.calc_EigStates(spin)

f_e = ge * spin.muB * B0 * spin.meVtoGHzConversion
spin.f_RF = f_e # use the eelectron frequency as the drive frequency

print(f"Electron carrier: {spin.f_RF:.6f} GHz")
print(
    "Nuclear quadrupole coefficients (MHz):",
    spin.Qvector[1] * spin.meVtoGHzConversion * 1e3,
)

## Initial state and readout

Prepare a fully polarized electron and an unpolarized nucleus:

$$
\rho_0=|+\rangle\langle+|\otimes\frac{\mathbb1_3}{3}.
$$

Use a fixed 1 GHz Rabi frequency as a numerical hard-pulse reference.
This is a comparison within the rotating-wave and electron-secular
approximations, not a proposed experimental drive strength.


In [ ]:
rho_e = np.diag([1.0, 0.0])
spin.psi0 = Qobj(np.kron(rho_e, np.eye(3) / 3))
spin.collapse_ops = []
spin.RabiRate = 1.0  # GHz; numerical hard-pulse reference

CC.RotFrameSecular(spin, spin.f_RF)
readout = [Qobj(spin.Sz_rot)]

## Nuclear frequencies and the quadrupole mixing

After the electron-secular approximation, the nuclear Hamiltonian
conditioned on $m_S=\pm1/2$ is

$$
\frac{H_n(m_S)}{h}
=
\begin{pmatrix}
K+\delta_m & 0 & K\eta\\
0 & -2K & 0\\
K\eta & 0 & K-\delta_m
\end{pmatrix},
\qquad
\delta_m=-\nu_N+m_S A_{\mathrm{iso}}.
$$

Its eigenvalues are $-2K$ and $K\pm r_m$, with

$$
r_m=\sqrt{\delta_m^2+(K\eta)^2}.
$$

The three nuclear energy gaps in each electron manifold are
$|3K-r_m|$, $|3K+r_m|$, and $2r_m$.

However, not every energy gap produces an ESEEM peak. In this particular
aligned-field model, $m_I=0$ remains a spectator. The quadrupole term
mixes only $m_I=+1$ and $m_I=-1$, giving the two modulation frequencies

$$
\nu_+=2r_{+1/2},\qquad \nu_-=2r_{-1/2}.
$$

The mixing parameter of this two-state subspace is

$$
k_{\mathrm{mix}}
=
\left(\frac{K\eta A_{\mathrm{iso}}}{r_{+1/2}r_{-1/2}}\right)^2.
$$

The spectator contributes one-third of the normalized signal.
Consequently, the envelopes below differ from the spin-1/2 nuclear
formulas used in the C13 example. They apply to this special aligned,
isotropic-hyperfine model, not to arbitrary spin-1 systems.

In [ ]:
nu_N = gamma_n * B0
delta = -nu_N + np.array([0.5, -0.5]) * A_iso
r_nuclear = np.hypot(delta, K * eta)

nu_plus, nu_minus = 2 * r_nuclear
k_mix = (K * eta * A_iso / np.prod(r_nuclear)) ** 2

for label, radius in zip(["+", "-"], r_nuclear, strict=True):
    gaps = np.array(
        [abs(3 * K - radius), abs(3 * K + radius), 2 * radius]
    )
    print(f"m_S = {label}1/2: nuclear energy gaps (MHz) = {gaps * 1e3}")

print(f"ESEEM frequencies: {nu_plus * 1e3:.6f}, {nu_minus * 1e3:.6f} MHz")
print(f"Mixing parameter: {k_mix:.6f}")

## Hahn echo

Simulate $\pi/2_x-\tau-\pi_x-\tau-\pi/2_x$.

For instantaneous pulses, the normalized envelope is

$$
V_2(\tau)
=
1-\frac{4k_{\mathrm{mix}}}{3}
\sin^2(\pi\nu_+\tau)\sin^2(\pi\nu_-\tau).
$$

The factor $2/3$ relative to the corresponding two-state nuclear
envelope accounts for the unmodulated spectator state.

The ideal spectrum versus $\tau$ contains $\nu_+$, $\nu_-$,
$|\nu_+-\nu_-|$, and $\nu_++\nu_-$.
The FFT coordinate is $\tau$, not $2\tau$.

In [ ]:
tau_array = np.arange(0.0, 10_000.0, 5.0)  # ns
_, hahn_result = CC.calc_HahnEcho_pluspihalf_Parallel(
    spin, tau_array=tau_array, e_ops=readout, n_jobs=1
)
hahn_simulated = 2 * hahn_result[:, 0]

sine_plus = np.sin(np.pi * nu_plus * tau_array)
sine_minus = np.sin(np.pi * nu_minus * tau_array)
hahn_ideal = 1 - 4 * k_mix / 3 * (sine_plus * sine_minus) ** 2

frequencies_hahn, spectrum_hahn = hf.calc_fft(
    times=tau_array, signal=hahn_simulated, window="hann"
)
_, spectrum_hahn_ideal = hf.calc_fft(
    times=tau_array, signal=hahn_ideal, window="hann"
)

hahn_lines = [nu_minus, abs(nu_plus - nu_minus), nu_plus, nu_plus + nu_minus]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))

axes[0].plot(tau_array / 1e3, hahn_simulated, label="Spinfinity")
axes[0].plot(tau_array / 1e3, hahn_ideal, "--", label="Ideal pulses")
axes[0].set(
    xlim=(0, 2),
    xlabel=r"$\tau$ ($\mu$s)",
    ylabel="Normalized Hahn signal",
)

axes[1].plot(frequencies_hahn * 1e3, abs(spectrum_hahn), label="Spinfinity")
axes[1].plot(
    frequencies_hahn * 1e3,
    abs(spectrum_hahn_ideal),
    "--",
    label="Ideal pulses",
)
for frequency in hahn_lines:
    axes[1].axvline(frequency * 1e3, color="gray", ls=":", alpha=0.6)

axes[1].set(
    xlim=(0, 16),
    xlabel="Frequency conjugate to tau (MHz)",
    ylabel="One-sided FFT magnitude",
)
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.tight_layout()
plt.show()

## Eight-step three-pulse ESEEM

Simulate $\pi/2-\tau-\pi/2-T-\pi/2-\tau-\pi/2$ with the
eight-step rephasing phase cycle. The last pulse performs population
readout.

For this model, the ideal normalized envelope is

$$
\begin{aligned}
V_3(\tau,T)=1-\frac{k_{\mathrm{mix}}}{6}\bigl[
 &(1-\cos(2\pi\nu_-\tau))
  (1-\cos(2\pi\nu_+(T+\tau)))\\
+&(1-\cos(2\pi\nu_+\tau))
  (1-\cos(2\pi\nu_-(T+\tau)))
\bigr].
\end{aligned}
$$

Choose $\tau=1/(2\nu_-)$ to maximize the ideal $\nu_+$ contribution.
Only the conditional frequencies $\nu_+$ and $\nu_-$ appear in the
ideal spectrum versus storage time $T$.

The reference envelopes follow from the effective two-state nuclear
subspace plus its one-third spectator contribution. Small differences
remain because the simulated pulses have finite duration.

In [ ]:
T_array = np.arange(0.0, 10_000.0, 5.0)  # ns
tau_fixed = 1 / (2 * nu_minus)

_, stimulated_result = CC.calc_3pulseESEEM_pluspihalf_PhaseCycled8(
    spin, T_array=T_array, tau=tau_fixed, e_ops=readout, n_jobs=1
)
stimulated_simulated = 4 * stimulated_result[:, 0]

stimulated_ideal = 1 - k_mix / 6 * (
    (1 - np.cos(2 * np.pi * nu_minus * tau_fixed))
    * (1 - np.cos(2 * np.pi * nu_plus * (T_array + tau_fixed)))
    + (1 - np.cos(2 * np.pi * nu_plus * tau_fixed))
    * (1 - np.cos(2 * np.pi * nu_minus * (T_array + tau_fixed)))
)

frequencies_stimulated, spectrum_stimulated = hf.calc_fft(
    times=T_array, signal=stimulated_simulated, window="hann"
)
_, spectrum_stimulated_ideal = hf.calc_fft(
    times=T_array, signal=stimulated_ideal, window="hann"
)

fig, axes = plt.subplots(1, 2, figsize=(11, 3.7))

axes[0].plot(T_array / 1e3, stimulated_simulated, label="Spinfinity")
axes[0].plot(T_array / 1e3, stimulated_ideal, "--", label="Ideal pulses")
axes[0].set(
    xlim=(0, 2),
    xlabel=r"$T$ ($\mu$s)",
    ylabel="Normalized stimulated signal",
)

axes[1].plot(
    frequencies_stimulated * 1e3,
    abs(spectrum_stimulated),
    label="Spinfinity",
)
axes[1].plot(
    frequencies_stimulated * 1e3,
    abs(spectrum_stimulated_ideal),
    "--",
    label="Ideal pulses",
)
for frequency, label in zip(
    [nu_plus, nu_minus], [r"$\nu_+$", r"$\nu_-$"], strict=True
):
    axes[1].axvline(frequency * 1e3, color="gray", ls=":", label=label)

axes[1].set(
    xlim=(0, 16),
    xlabel="Frequency conjugate to T (MHz)",
    ylabel="One-sided FFT magnitude",
)
for ax in axes:
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

fig.tight_layout()
plt.show()

print(f"Fixed tau: {tau_fixed:.3f} ns")

## Background literature

- [Lee, Doan and Hoffman (1999), General Analysis of 14N (I = 1)
  Electron Spin Echo Envelope Modulation](https://doi.org/10.1006/jmre.1999.1803).
  General theory of spin-1 nuclear ESEEM.
- [Gunnar Jeschke, ESEEM and HYSCORE](https://chem.libretexts.org/Bookshelves/Physical_and_Theoretical_Chemistry_Textbook_Maps/Electron_Paramagnetic_Resonance_(Jenschke)/08:_Measurement_of_Small_Hyperfine_Couplings/8.02:_ESEEM_and_HYSCORE).
  Background on stimulated echoes and nuclear modulation.
- [EasySpin: nuclear quadrupole interaction](https://easyspin.org/documentation/hamiltonian.html#nqi).
  Quadrupole tensors, coupling constants, and the electric field gradient.